#### The Retail UCI Dataset

In [12]:
import pandas as pd
path = '../data/online_retail.xlsx'
df = pd.read_excel(path)

- `InvoiceNo`: a 6-digit integral number uniquely assigned to each transaction. If this code starts with letter 'c', it indicates a cancellation.

- `StockCode`: a 5-digit integral number uniquely assigned to each distinct products.

- `Description`: product name.

- `Quantity`: The quantities of each product per transaction.

- `InvoiceDate`: the day and time when each transaction was generated.

- `UnitPrice`: product price per unit.

- `CustomerID`: 5-digit integral number uniquely assigned to each customer.

- `Country`: the name of the country where each customer resides.

In [ ]:
df.dtypes

In [ ]:
df.shape

In [ ]:
n = 9288 # number of cancel InvoiceNo
df_a = df[df['InvoiceNo'].astype(str).str.startswith('C', na=False)]

In [ ]:
df_b = df[df['Quantity'] < 0]

In [ ]:
df_diff = (
    df_b.merge(df_a.drop_duplicates(), how='left', indicator=True)
    .query('_merge == "left_only"')
    .drop(columns='_merge')
)

In [ ]:
df_diff

In [ ]:
col = ['InvoiceNo', 'StockCode', 'Description', 'CustomerID']
df[col].nunique()

In [ ]:
df.isna().sum()

- There are more than 135080/541909 records that do not have the id of customer and some invoices do not contains product's description. These records should be deeply explore in the next part of this notebook.

- The number of unique `InvoiceNo` << the numbers of rows in UCI datasets. In addition, the `CustomerID` unique number exists in this dataset is less than `InvoiceNo`. We can conclude that a invoice could contain more than one customer.

- `Quantity` column has a significant number of "errorneous" observers, which > 10k records have the negative quantity value. Especially, 90% of these rows have cancelled invoice. This problem will be analysed in the next step of EDA.

In [ ]:
df[['UnitPrice', 'Quantity']].describe()

Records with the description "Adjust bad debt" contain anomalous UnitPrice values, including unusually large positive and negative amounts. These records represent accounting adjustments rather than actual retail transactions and may distort the Monetary component of RFM analysis.

Therefore, these records are excluded from the dataset used for customer segmentation, as they do not reflect genuine purchasing behavior and have missing CustomerID values that prevent reliable customer-level attribution. The original dataset is preserved for traceability.

##### `StockCode`

In [ ]:
# StockCode có đúng 5 chữ số
is_5_digit = df['StockCode'].astype(str).str.match(r'^\d{5}$')

# StockCode có 5 chữ số kèm theo 1 số chữ cái
is_variant_product = df['StockCode'].astype(str).str.match(r'^\d{5}[a-zA-Z]{1,2}$')

variants = df[is_variant_product]
print("Số mã biến thể (5 số + chữ cái):", variants['StockCode'].nunique())

# 2. Các mã dịch vụ / dị biệt
special_codes = df[~(is_5_digit | is_variant_product)]
print("Các mã dịch vụ đặc biệt:")
print(special_codes[['StockCode', 'Description']].drop_duplicates())

- Standard & Variant Products: 5 digits (e.g., `22423`) or 5 digits with letter suffixes (e.g., `85123A`) representing color/size variants $\rightarrow$ **Keep**.
- Non-Product Codes: Service, fee, or manual entries (e.g., `POST`, `DOT`, `C2`, `D`, `M`, `BANK CHARGES`, `AMAZONFEE`, `gift_...`, `B`)
    + Shipping (`POST`), platform fees, and discounts represent operational overhead, not customer merchandise demand.
    + Internal accounting adjustments (`B`, `M`) and heavy fees skew `Monetary` value and inflate `Frequency`.
    + Prevents Algorithm Bias: Prevents postage (`POST`) from being falsely flagged as a "top-selling product" in basket analysis/clustering.
    + Gift cards (`gift_...`) are payment instruments, not consumable goods.
- These stock codes will eliminated from the dataset. With the group of stock codes like `DCGS...` and `PAD` should be check carefully to determine that whether this group indicates a valid product from customers.

In [ ]:
special_codes[
    (
        special_codes['StockCode']
        .astype(str) 
        .str.startswith("DCGS")
    ) | 
    (
        special_codes['StockCode']
        .astype(str)
        .str.startswith('PADS')
    )
]

In [ ]:
with pd.option_context('display.max_rows', None):
    display(df['Description'].value_counts())

In [ ]:
with pd.option_context('display.max_rows', None):
    display(df[df['UnitPrice'] == 0].loc[:, 'Description'].value_counts())

- Lowercase & Symbol Descriptions (`check`, `damaged`, `found`, `?`): Standard product names in the catalog are strictly UPPERCASE. Lowercase descriptions and question marks represent internal warehouse memos (e.g., stock checks, damaged/unsaleable goods, inventory adjustments) rather than customer purchases.
- 100% Overlap with `UnitPrice == 0`: Every single lowercase/memo description occurs exclusively when `UnitPrice == 0` (e.g., all 159 `check` and 47 `?` records have a price of `0`).
- Uppercase Products with `UnitPrice == 0`: Represent either unannotated stock adjustments (`CustomerID` is `NaN`) or free promotional/replacement items (`CustomerID` is present), neither of which contribute to revenue.
- The `StockCode` with value `PADS` or starts with `DCGS...` appears with missing `CustomerID`.

In [ ]:
df = df[(df['UnitPrice'] != 0) & (is_5_digit | is_variant_product)]

##### Missing value handle

Since records with missing `CustomerID` values cannot be reliably linked to individual customers for subsequent profiling and tracking, these records were excluded from the dataset.

In [ ]:
# Dataset contains only missing CustomerID
df_clean = df[~df['CustomerID'].isna()]

##### Negative Quanity and Cancelled Invoice

In [ ]:
df_clean[(df_clean['InvoiceNo'].astype(str).str.startswith('C')) & (df_clean['Quantity'] < 0)]

In [ ]:
df_clean[df_clean['Quantity'] < 0]

##### Cleaning Data Decision

- Remove all records with missing values in `CustomerID`.
- Remove records where `UnitPrice` is zero or negative.
- Remove records whose `Description` is not written entirely in uppercase letters.
- Treat invoices with an `InvoiceNo` prefixed by `C` and a negative `Quantity` as potential indicators of customer return behavior.

In [13]:
# Remove missing CustomerID
df.dropna(subset=['CustomerID'], inplace=True)

# Remove error UnitPrice
df = df[df['UnitPrice'] > 0]

# Remove strange StockCode
is_digit_5 = df['StockCode'].astype(str).str.match(r'^\d{5}$')
is_variant_product = is_variant_product = df['StockCode'].astype(str).str.match(r'^\d{5}[a-zA-Z]{1,2}$')

df = df[is_digit_5 | is_variant_product]

# Remove negative quantity
df = df[df['Quantity'] >= 0]

In [18]:
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)
df['CustomerID'] = df['CustomerID'].astype('Int64').astype(str)
df['Country'] = df['Country'].astype('category')

In [19]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebook":
    project_root = project_root.parent

# Make a new folder
cleaned_dir = project_root / "data" / "cleaned"
cleaned_dir.mkdir(parents=True, exist_ok=True)

# Save file
output = cleaned_dir / "retail_online.parquet"
df.to_parquet(output, index=False)